# LSTM-arithmetic

## Dataset
- [Arithmetic dataset](https://drive.google.com/file/d/1cMuL3hF9jefka9RyF4gEBIGGeFGZYHE-/view?usp=sharing)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ! pip install seaborn
# ! pip install opencc
# ! pip install -U scikit-learn

import numpy as np
import pandas as pd
import torch
import torch.nn
import torch.nn.utils.rnn
import torch.utils.data
import matplotlib.pyplot as plt
import seaborn as sns
#import opencc
import os
from sklearn.model_selection import train_test_split

data_path = '/content/drive/MyDrive/nlp-hw2'

In [3]:
df_train = pd.read_csv(os.path.join(data_path, 'arithmetic_train.csv'))
df_eval = pd.read_csv(os.path.join(data_path, 'arithmetic_eval.csv'))
df_train.head()

,src,tgt
0,(40*0)-18=,-18
1,6+4+31=,41
2,4*11+13=,57
3,44*39*26=,44616
4,(39*42)+27=,1665


In [4]:
# transform the input data to string
df_train['tgt'] = df_train['tgt'].apply(lambda x: str(x))
df_train['src'] = df_train['src'].add(df_train['tgt'])
df_train['len'] = df_train['src'].apply(lambda x: len(x))

df_eval['tgt'] = df_eval['tgt'].apply(lambda x: str(x))

# Build Dictionary
 - The model cannot perform calculations directly with plain text.
 - Convert all text (numbers/symbols) into numerical representations.
 - Special tokens
    - '&lt;pad&gt;'
        - Each sentence within a batch may have different lengths.
        - The length is padded with '&lt;pad&gt;' to match the longest sentence in the batch.
    - '&lt;eos&gt;'
        - Specifies the end of the generated sequence.
        - Without '&lt;eos&gt;', the model will not know when to stop generating.

In [5]:
char_to_id = {}
id_to_char = {}

# write your code here
# Build a dictionary and give every token in the train dataset an id
# The dictionary should contain <eos> and <pad>
# char_to_id is to conver charactors to ids, while id_to_char is the opposite

special_tokens = ['<pad>', '<eos>']
arithmetic_chars = ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '+', '-', '*', '(', ')', '=']
vocab = special_tokens + arithmetic_chars

char_to_id = {ch: idx for idx, ch in enumerate(vocab)}
id_to_char = {idx: ch for idx, ch in enumerate(vocab)}

vocab_size = len(char_to_id)
print('Vocab size{}'.format(vocab_size))

Vocab size18


# Data Preprocessing
 - The data is processed into the format required for the model's input and output. (End with \<eos\> token)


In [6]:
# Write your code here
pad_id = char_to_id['<pad>']
eos_id = char_to_id['<eos>']

srcs = df_train['src'].values
tgts = df_train['tgt'].values

char_id_list = []
label_id_list = []

for s,t in zip(srcs,tgts):
    c_ids = [char_to_id[c] for c in s]
    prefix_len = len(s) - len(t)-1
    l_ids = [pad_id] * prefix_len + [char_to_id[c] for c in t] + [eos_id]
    char_id_list.append(c_ids)
    label_id_list.append(l_ids)

df_train['char_id_list'] = char_id_list
df_train['label_id_list'] = label_id_list

df_train.head()

,src,tgt,len,char_id_list,label_id_list
0,(40*0)-18=-18,-18,13,"[15, 6, 2, 14, 2, 16, 13, 3, 10, 17, 13, 3, 10]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 13, 3, 10, 1]"
1,6+4+31=41,41,9,"[8, 12, 6, 12, 5, 3, 17, 6, 3]","[0, 0, 0, 0, 0, 0, 6, 3, 1]"
2,4*11+13=57,57,10,"[6, 14, 3, 3, 12, 3, 5, 17, 7, 9]","[0, 0, 0, 0, 0, 0, 0, 7, 9, 1]"
3,44*39*26=44616,44616,14,"[6, 6, 14, 5, 11, 14, 4, 8, 17, 6, 6, 8, 3, 8]","[0, 0, 0, 0, 0, 0, 0, 0, 6, 6, 8, 3, 8, 1]"
4,(39*42)+27=1665,1665,15,"[15, 5, 11, 14, 6, 4, 16, 12, 4, 9, 17, 3, 8, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 8, 8, 7, 1]"


# Hyper Parameters

|Hyperparameter|Meaning|Value|
|-|-|-|
|`batch_size`|Number of data samples in a single batch|64|
|`epochs`|Total number of epochs to train|10|
|`embed_dim`|Dimension of the word embeddings|256|
|`hidden_dim`|Dimension of the hidden state in each timestep of the LSTM|256|
|`lr`|Learning Rate|0.001|
|`grad_clip`|To prevent gradient explosion in RNNs, restrict the gradient range|1|

In [7]:
batch_size = 128
epochs = 14
embed_dim = 256
hidden_dim = 512
lr = 0.001
grad_clip = 1

# Data Batching
- Use `torch.utils.data.Dataset` to create a data generation tool called  `dataset`.
- The, use `torch.utils.data.DataLoader` to randomly sample from the `dataset` and group the samples into batches.

- Example: 1+2-3=0
    - Model input: 1 + 2 - 3 = 0
    - Model output: / / / / / 0 &lt;eos&gt;  (the '/' can be replaced with &lt;pad&gt;)
    - The key for the model's output is that the model does not need to predict the next character of the previous part. What matters is that once the model sees '=', it should start generating the answer, which is '0'. After generating the answer, it should also generate&lt;eos&gt;

In [8]:
class Dataset(torch.utils.data.Dataset):
    def __init__(self, sequences):
        self.sequences = sequences

    def __len__(self):
        # return the amount of data
        return len(self.sequences)# Write your code here

    def __getitem__(self, index):
        # Extract the input data x and the ground truth y from the data
        x = self.sequences.iloc[index]['char_id_list']# Write your code here
        y = self.sequences.iloc[index]['label_id_list']# Write your code here
        return x, y

# collate function, used to build dataloader
def collate_fn(batch):
    batch_x = [torch.tensor(data[0]) for data in batch]
    batch_y = [torch.tensor(data[1]) for data in batch]
    batch_x_lens = torch.LongTensor([len(x) for x in batch_x])
    batch_y_lens = torch.LongTensor([len(y) for y in batch_y])

    # Pad the input sequence
    pad_batch_x = torch.nn.utils.rnn.pad_sequence(batch_x,
                                                  batch_first=True,
                                                  padding_value=char_to_id['<pad>'])

    pad_batch_y = torch.nn.utils.rnn.pad_sequence(batch_y,
                                                  batch_first=True,
                                                  padding_value=char_to_id['<pad>'])

    return pad_batch_x, pad_batch_y, batch_x_lens, batch_y_lens

In [9]:
ds_train = Dataset(df_train[['char_id_list', 'label_id_list']])

In [10]:
# Build dataloader of train set and eval set, collate_fn is the collate function
dl_train = torch.utils.data.DataLoader(ds_train, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)# Write your code here




# Model Design

## Execution Flow
1. Convert all characters in the sentence into embeddings.
2. Pass the embeddings through an LSTM sequentially.
3. The output of the LSTM is passed into another LSTM, and additional layers can be added.
4. The output from all time steps of the final LSTM is passed through a Fully Connected layer.
5. The character corresponding to the maximum value across all output dimensions is selected as the next character.

## Loss Function
Since this is a classification task, Cross Entropy is used as the loss function.

## Gradient Update
Adam algorithm is used for gradient updates.

In [11]:
class CharRNN(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(CharRNN, self).__init__()
        # Write your code here
        # Design your own model, which should contain embedding layer, several hidden layers and a linear layer

        self.embedding = torch.nn.Embedding(num_embeddings=vocab_size,
                            embedding_dim=embed_dim,
                            padding_idx=char_to_id['<pad>'])

        self.lstm = torch.nn.LSTM(input_size=embed_dim,
                        hidden_size=hidden_dim,
                        num_layers=3,
                        dropout=0.1,
                        batch_first=True)

        self.linear = torch.nn.Sequential(torch.nn.Linear(in_features=hidden_dim,
                                    out_features=hidden_dim),
                                    torch.nn.ReLU(),
                                    torch.nn.Linear(in_features=hidden_dim,
                                    out_features=vocab_size))

    def forward(self, batch_x, batch_x_lens):
        return self.encoder(batch_x, batch_x_lens)

    # The forward pass of the model
    def encoder(self, batch_x, batch_x_lens):
        batch_x = self.embedding(batch_x)

        batch_x = torch.nn.utils.rnn.pack_padded_sequence(batch_x,
                                  batch_x_lens,
                                  batch_first=True,
                                  enforce_sorted=False)

        batch_x, _ = self.lstm(batch_x)

        batch_x, _ = torch.nn.utils.rnn.pad_packed_sequence(batch_x,batch_first=True)

        batch_x = self.linear(batch_x)

        return batch_x

    def generator(self, start_char, max_len=200):

        char_list = [char_to_id[c] for c in start_char]

        next_char = None

        while len(char_list) < max_len:
            # Write your code here
            # Pack the char_list to tensor
            # Input the tensor to the embedding layer, RNN layers, linear respectively
            input_tensor = torch.tensor([char_list], dtype=torch.long, device=next(self.parameters()).device)
            out = self.embedding(input_tensor)
            out, _ = self.lstm(out)
            out = self.linear(out)
            y = out[0, -1, :] # Obtain the next token prediction y

            next_char = torch.argmax(y).item() # Use argmax function to get the next token prediction

            if next_char == char_to_id['<eos>']:
                break

            char_list.append(next_char)

        return [id_to_char[ch_id] for ch_id in char_list]

In [12]:
torch.manual_seed(2)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')# Write your code here. Specify a device (cuda or cpu)

model = CharRNN(vocab_size,
                embed_dim,
                hidden_dim)

In [13]:
criterion = torch.nn.CrossEntropyLoss(ignore_index=char_to_id['<pad>'])# Write your code here. Cross-entropy loss function. The loss function should ignore <pad>
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)# Write your code here. Use Adam or AdamW for Optimizer


# Training
1. The outer `for` loop controls the `epoch`
    1. The inner `for` loop uses `data_loader` to retrieve batches.
        1. Pass the batch to the `model` for training.
        2. Compare the predicted results `batch_pred_y` with the true labels `batch_y` using Cross Entropy to calculate the loss `loss`
        3. Use `loss.backward` to automatically compute the gradients.
        4. Use `torch.nn.utils.clip_grad_value_` to limit the gradient values between `-grad_clip` &lt; and &lt; `grad_clip`.
        5. Use `optimizer.step()` to update the model (backpropagation).
2.  After every `1000` batches, output the current loss to monitor whether it is converging.

In [14]:
drive_checkpoint_dir = '/content/drive/MyDrive/nlp-hw2'
drive_checkpoint_path = os.path.join(drive_checkpoint_dir, 'checkpoint.pt')
local_checkpoint_path = 'checkpoint.pt'

In [15]:
from tqdm import tqdm
from copy import deepcopy
model = model.to(device)
model.train()
i = 0
for epoch in range(1, epochs+1):
    # The process bar
    bar = tqdm(dl_train, desc=f"Train epoch {epoch}")
    for batch_x, batch_y, batch_x_lens, batch_y_lens in bar:
        # Write your code here
        # Clear the gradient
        optimizer.zero_grad()

        batch_pred_y = model(batch_x.to(device), batch_x_lens)

        # Write your code here
        # Input the prediction and ground truths to loss function
        loss = criterion(batch_pred_y.view(-1, vocab_size), batch_y.to(device).view(-1))
        # Back propagation
        loss.backward()

        torch.nn.utils.clip_grad_value_(model.parameters(), grad_clip) # gradient clipping

        # Write your code here
        # Optimize parameters in the model
        optimizer.step()

        i+=1
        if i%50==0:
            bar.set_postfix(loss = loss.item())

    # Evaluate your model
    matched = 0
    total = 0
    bar_eval = tqdm(df_eval.iterrows(), desc=f"Validation epoch {epoch}")
    for _, row in bar_eval:
        batch_x = row['src']
        batch_y = row['tgt']

        prediction = model.generator(batch_x)# An example of using generator: model.generator('1+1=')

        # Write your code here. Input the batch_x to the model and generate the predictions
        # Write your code here.
        # Check whether the prediction match the ground truths
        # Compute exact match (EM) on the eval dataset
        # EM = correct/total
        pred_ans = ''.join(prediction[len(batch_x):])
        if pred_ans == str(batch_y):
            matched += 1
        total += 1


    print(matched/total)

Train epoch 1: 100%|██████████| 7813/7813 [02:18<00:00, 56.26it/s, loss=0.496]
Validation epoch 1: 10000it [00:49, 200.60it/s]


0.4494


Train epoch 2: 100%|██████████| 7813/7813 [02:16<00:00, 57.30it/s, loss=0.346]
Validation epoch 2: 10000it [00:49, 202.37it/s]


0.5615


Train epoch 3: 100%|██████████| 7813/7813 [02:17<00:00, 56.95it/s, loss=0.31]
Validation epoch 3: 10000it [00:50, 199.36it/s]


...

0.8667


Train epoch 13: 100%|██████████| 7813/7813 [02:17<00:00, 56.97it/s, loss=0.111]
Validation epoch 13: 10000it [00:49, 201.34it/s]


0.8753


Train epoch 14: 100%|██████████| 7813/7813 [02:17<00:00, 56.86it/s, loss=0.118]
Validation epoch 14: 10000it [00:49, 201.26it/s]

0.8701


In [16]:
torch.save(model.state_dict(), '/content/drive/MyDrive/nlp-hw2/checkpoint_lstm_14.pt')
print('ok')

ok


Running environment: colab

Python version: colab

##### 1. Describe your model architecture, data preprocessing methods, and hyperparameter settings, and justify your choices.  (5%)

Answer:

##### 2. If you use 2-layer LSTM or GRU instead of RNN, what will happen to the quality of your answer generation? Why? (10%)

Answer:

##### 3. If we construct an training set using three-digit numbers while the evaluation set is constructed from two-digit numbers, what will happen to the quality of your answer generation? (10%)

Answer:

##### 4. If we construct a training set that includes 20% incorrect answers, how will this affect the quality of the generated responses? Present some examples. (10%)

Answer:

##### 5. Why do we need gradient clipping during training? (5%)

Answer:

##### … Anything that can strengthen your report. (5%)

Answer:

##### Generative AI Usage Statement

Answer:
